<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [1]:
import torch
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [2]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Создаем датасет для предобработки данных

In [3]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Загружаем данные и разметку для объекта с индексом `idx`.

        labels: List[int] Набор классов для каждого ббокса,
        boxes: List[List[int]] Набор ббоксов в формате (x_min, y_min, w, h).
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        target['boxes'] = torch.tensor(np.array(boxes), dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [4]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        # Добавляй сюда свои аугментации при необходимости!
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    # Раскомментируй, если аугментации изменяют ббоксы.
    # Не забудь указать верный формат для ббоксов.
    # bbox_params=A.BboxParams(format='coco', label_fields=['labels'])
)

test_transform = A.Compose(
    [
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ]
)

Не забываем инициализировать наш датасет

In [5]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import numpy as np
from torchvision.ops import box_iou
class Backbone(nn.Module):
    def __init__(self, backbone_name='resnet50', unfreeze_last=0, pretrained=True):
        super().__init__()
        weights = 'IMAGENET1K_V1' if pretrained else None
        resnet = torchvision.models.resnet50(weights=weights)

        # Убираем avgpool и fc, оставляем только сверточную часть
        self.conv1 = resnet.conv1
        self.bn1   = resnet.bn1
        self.relu  = resnet.relu
        self.maxpool = resnet.maxpool

        self.layer1 = resnet.layer1  # C2
        self.layer2 = resnet.layer2  # C3
        self.layer3 = resnet.layer3  # C4
        self.layer4 = resnet.layer4  # C5

        self.out_channels = [512, 1024, 2048]  # каналы для C3, C4, C5

        # Замораживаем все, кроме последних unfreeze_last блоков
        self._freeze_unfreeze(unfreeze_last)

    def _freeze_unfreeze(self, k):
        # Заморозить все параметры
        for param in self.parameters():
            param.requires_grad = False
        # Разморозить последние k блоков (layer4 – самый последний)
        all_blocks = [self.layer1, self.layer2, self.layer3, self.layer4]
        for block in all_blocks[-k:]:
            for param in block.parameters():
                param.requires_grad = True

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        c2 = self.layer1(x)   # /4
        c3 = self.layer2(c2)  # /8
        c4 = self.layer3(c3)  # /16
        c5 = self.layer4(c4)  # /32

        # Возвращаем C3, C4, C5 (от высокого разрешения к низкому)
        return [c3, c4, c5]

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [7]:
import torch.nn as nn

class Neck(nn.Module):
    def __init__(self, in_channels_list, out_channels=256):
        super().__init__()
        # Латеральные свёртки 1x1 для приведения каналов к out_channels
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(in_ch, out_channels, kernel_size=1)
            for in_ch in in_channels_list
        ])
        # Выходные свёртки 3x3 после суммирования
        self.fpn_convs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
            for _ in in_channels_list
        ])

    def forward(self, features):
        # features: [C3, C4, C5] (высокое разрешение → низкое)
        # Переворачиваем, чтобы начать с самого семантически сильного слоя (C5)
        laterals = [
            conv(f) for conv, f in zip(self.lateral_convs, features[::-1])
        ]
        # Top-down путь с Nearest Upsampling
        fpn_outputs = []
        x = laterals[0]                      # P5
        fpn_outputs.append(self.fpn_convs[0](x))

        for i in range(1, len(laterals)):
            x = F.interpolate(x, scale_factor=2, mode='nearest')
            x = x + laterals[i]             # P4, затем P3
            fpn_outputs.append(self.fpn_convs[i](x))

        # Возвращаем в порядке: P3, P4, P5 (высокое разрешение → низкое)
        return fpn_outputs[::-1]

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [8]:
class DecoupledHead(nn.Module):
    def __init__(self, in_channels, num_classes, num_anchors=1):
        super().__init__()
        self.num_classes = num_classes
        self.num_anchors = num_anchors

        # Ветка классификации
        self.cls_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
        )
        # Ветка регрессии
        self.reg_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
        )
        # Предсказания
        self.cls_pred = nn.Conv2d(in_channels, num_anchors * num_classes, 1)
        self.reg_pred = nn.Conv2d(in_channels, num_anchors * 4, 1)
        self.obj_pred = nn.Conv2d(in_channels, num_anchors * 1, 1)   # objectness

    def forward(self, x):
        cls_feat = self.cls_conv(x)
        reg_feat = self.reg_conv(x)

        cls_score = self.cls_pred(cls_feat)   # (B, num_anchors * C, H, W)
        bbox_pred = self.reg_pred(reg_feat)   # (B, num_anchors * 4, H, W)
        obj_score = self.obj_pred(reg_feat)   # (B, num_anchors * 1, H, W)
        return cls_score, bbox_pred, obj_score


class Head(nn.Module):
    """Многоуровневая голова – по одной DecoupledHead на каждый выход FPN."""
    def __init__(self, in_channels, num_classes, num_anchors=1, num_levels=3):
        super().__init__()
        self.heads = nn.ModuleList([
            DecoupledHead(in_channels, num_classes, num_anchors)
            for _ in range(num_levels)
        ])

    def forward(self, features):
        # features – список [P3, P4, P5]
        all_cls, all_bbox, all_obj = [], [], []
        for feat, head in zip(features, self.heads):
            cls, bbox, obj = head(feat)
            all_cls.append(cls)
            all_bbox.append(bbox)
            all_obj.append(obj)
        return all_cls, all_bbox, all_obj

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [9]:
class Detector(nn.Module):
    def __init__(self, num_classes, backbone_unfreeze=2, anchor_sizes=...):
        super().__init__()
        self.backbone = Backbone(unfreeze_last=backbone_unfreeze)
        self.neck = Neck(in_channels_list=self.backbone.out_channels, out_channels=256)
        self.head = Head(in_channels=256, num_classes=num_classes)



    def forward(self, x):
        feats = self.backbone(x)
        feats = self.neck(feats)
        cls, bbox, obj = self.head(feats)
        return cls, bbox, obj

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [10]:
def TAL_assigner(cls_scores, bbox_preds, anchors_list, gt_boxes, gt_labels,
                 alpha=6.0, beta=1.0, top_k=5, bg_class=0):
    """
    cls_scores: список тензоров [B, num_anchors*C, H, W] (логиты, будут пропущены через sigmoid)
    bbox_preds: список тензоров [B, num_anchors*4, H, W] (смещения)
    anchors_list: список тензоров [num_anchors, 4] для каждого уровня (формат xyxy)
    gt_boxes: список тензоров для каждого изображения в batch, каждый (N, 4) xyxy
    gt_labels: список тензоров для каждого изображения, каждый (N,)
    bg_class: индекс фонового класса (например, 0)

    Возвращает:
        target_labels, target_bboxes, pos_mask (флаги положительных примеров)
        для всего batch – конкатенированные по всем уровням.
    """
    B = cls_scores[0].size(0)
    device = cls_scores[0].device
    num_classes = cls_scores[0].size(1) // anchors_list[0].size(0)  # кол-во якорей

    # Собираем все якоря и предсказания по уровням
    all_anchors = []
    all_cls_probs = []
    all_bbox_deltas = []
    for lvl, (cls, bbox, anchors) in enumerate(zip(cls_scores, bbox_preds, anchors_list)):
        N_anch, _ = anchors.shape
        # cls: (B, N_anch*C, H, W) -> (B, N_anch, C, H*W)
        C = num_classes
        B, _, H, W = cls.shape
        cls = cls.view(B, N_anch, C, H, W).permute(0, 1, 3, 4, 2).reshape(B, -1, C)
        # bbox: (B, N_anch*4, H, W) -> (B, N_anch, 4, H*W)
        bbox = bbox.view(B, N_anch, 4, H, W).permute(0, 1, 3, 4, 2).reshape(B, -1, 4)
        anchors_exp = anchors.view(1, N_anch, 1, 1, 4).expand(B, N_anch, H, W, 4).reshape(B, -1, 4)
        all_anchors.append(anchors_exp)
        all_cls_probs.append(cls.sigmoid())  # вероятности
        all_bbox_deltas.append(bbox)

    all_anchors = torch.cat(all_anchors, dim=1)        # (B, total_anchors, 4)
    all_cls_probs = torch.cat(all_cls_probs, dim=1)    # (B, total_anchors, C)
    all_bbox_deltas = torch.cat(all_bbox_deltas, dim=1)# (B, total_anchors, 4)

    total_anchors = all_anchors.shape[1]
    target_labels = torch.full((B, total_anchors), bg_class, device=device, dtype=torch.long)
    target_bboxes = torch.zeros(B, total_anchors, 4, device=device)
    pos_mask = torch.zeros(B, total_anchors, device=device, dtype=torch.bool)


    pred_boxes = all_bbox_deltas  # упрощение

    for i in range(B):
        if gt_boxes[i] is None or len(gt_boxes[i]) == 0:
            continue
        gt_box = gt_boxes[i]          # (M, 4)
        gt_label = gt_labels[i]       # (M,)
        M = gt_box.size(0)

        # 1. Вычисление t = s^\alpha * u^\beta
        # s – classification score соответствующего класса GT
        cls_prob = all_cls_probs[i]   # (total_anchors, C)
        # индекс класса GT для каждого предсказания
        gt_cls_prob = cls_prob[:, gt_label]  # (total_anchors, M)

        # IoU между pred_boxes и gt_boxes
        ious = box_iou(pred_boxes[i], gt_box)  # (total_anchors, M)

        t = gt_cls_prob.pow(alpha) * ious.pow(beta)  # (total_anchors, M)

        # 2. Фильтрация: центр анкора должен быть внутри GT
        anchors = all_anchors[i]  # (total_anchors, 4) xyxy
        a_cx = (anchors[:, 0] + anchors[:, 2]) / 2
        a_cy = (anchors[:, 1] + anchors[:, 3]) / 2
        inside_mask = torch.zeros(total_anchors, M, dtype=torch.bool, device=device)
        for j in range(M):
            gt = gt_box[j]
            inside_mask[:, j] = (a_cx >= gt[0]) & (a_cx <= gt[2]) & (a_cy >= gt[1]) & (a_cy <= gt[3])
        t[~inside_mask] = -1e9  # исключаем

        # 3. Выбор top_k для каждого GT и разрешение конфликтов
        assigned_gt_idx = torch.full((total_anchors,), -1, device=device, dtype=torch.long)
        for j in range(M):
            topk_vals, topk_idx = torch.topk(t[:, j], top_k)
            # Если предсказание ещё не назначено, назначаем текущему GT
            for idx in topk_idx:
                if assigned_gt_idx[idx] == -1:
                    assigned_gt_idx[idx] = j
                else:
                    # Уже назначено – оставляем GT с большим IoU
                    prev_j = assigned_gt_idx[idx]
                    if ious[idx, j] > ious[idx, prev_j]:
                        assigned_gt_idx[idx] = j

        # Формируем таргеты
        pos = assigned_gt_idx >= 0
        pos_mask[i] = pos
        assigned_gt = assigned_gt_idx[pos]
        target_labels[i, pos] = gt_label[assigned_gt]
        target_bboxes[i, pos] = gt_box[assigned_gt]

    return target_labels, target_bboxes, pos_mask

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [11]:
from torchvision.ops import distance_box_iou_loss

In [12]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [13]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [14]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 1.0246384143829346


In [17]:
def diou_loss(pred_boxes, gt_boxes):
    """
    pred_boxes: (N, 4) xyxy
    gt_boxes:   (N, 4) xyxy
    Возвращает усреднённый DIoU loss (скаляр)
    """
    # Площади
    area_pred = (pred_boxes[:, 2] - pred_boxes[:, 0]) * (pred_boxes[:, 3] - pred_boxes[:, 1])
    area_gt   = (gt_boxes[:, 2] - gt_boxes[:, 0]) * (gt_boxes[:, 3] - gt_boxes[:, 1])

    # Пересечение
    x1 = torch.max(pred_boxes[:, 0], gt_boxes[:, 0])
    y1 = torch.max(pred_boxes[:, 1], gt_boxes[:, 1])
    x2 = torch.min(pred_boxes[:, 2], gt_boxes[:, 2])
    y2 = torch.min(pred_boxes[:, 3], gt_boxes[:, 3])
    inter_w = (x2 - x1).clamp(min=0)
    inter_h = (y2 - y1).clamp(min=0)
    inter = inter_w * inter_h

    # Объединение
    union = area_pred + area_gt - inter
    iou = inter / (union + 1e-7)  # (N,)

    # Центры
    cx_pred = (pred_boxes[:, 0] + pred_boxes[:, 2]) / 2
    cy_pred = (pred_boxes[:, 1] + pred_boxes[:, 3]) / 2
    cx_gt   = (gt_boxes[:, 0] + gt_boxes[:, 2]) / 2
    cy_gt   = (gt_boxes[:, 1] + gt_boxes[:, 3]) / 2
    center_dist2 = (cx_pred - cx_gt)**2 + (cy_pred - cy_gt)**2

    # Диагональ охватывающего прямоугольника
    x1_c = torch.min(pred_boxes[:, 0], gt_boxes[:, 0])
    y1_c = torch.min(pred_boxes[:, 1], gt_boxes[:, 1])
    x2_c = torch.max(pred_boxes[:, 2], gt_boxes[:, 2])
    y2_c = torch.max(pred_boxes[:, 3], gt_boxes[:, 3])
    diag2 = (x2_c - x1_c)**2 + (y2_c - y1_c)**2 + 1e-7

    loss = 1 - iou + center_dist2 / diag2
    return loss.mean()

In [18]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?


### 1. Импорты и фиксация формата xyxy


In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
from torchvision.ops import box_convert, box_iou, nms, distance_box_iou_loss
from torchmetrics.detection import MeanAveragePrecision
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
import numpy as np
from PIL import Image
import io
from tqdm import tqdm
import math
from copy import deepcopy

### 2. Датасет (xyxy формат)

In [20]:
class HaloDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row['bytes'])).convert('RGB')
        image = np.array(image)

        labels = [row['category']] if isinstance(row['category'], int) else row['category']
        labels = [l - 1 for l in labels]  # классы с 0
        boxes = row['bbox'].tolist()      # COCO формат: xywh

        # Преобразуем в xyxy для аугментаций и дальнейшей работы
        bboxes = []
        for b in boxes:
            x, y, w, h = b
            bboxes.append([x, y, x+w, y+h])

        if self.transform:
            transformed = self.transform(image=image, bboxes=bboxes, labels=labels)
            image = transformed['image']
            bboxes = transformed['bboxes']
            labels = transformed['labels']
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            # нормализация
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
            image = (image - mean) / std

        target = {
            'boxes': torch.tensor(bboxes, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.int64),
            'image_id': row['image_id']
        }
        return image, target

def collate_fn(batch):
    images, targets = zip(*batch)
    images = torch.stack(images)
    return images, list(targets)

### 3. Аугментации

In [21]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

test_transform = A.Compose([
    A.Resize(640, 640),
    A.Normalize(mean=mean, std=std),
    ToTensorV2()
], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

### 4. Генератор якорей (для 3 уровней FPN)

In [22]:
class AnchorGenerator(nn.Module):
    def __init__(self, strides=[8, 16, 32], sizes=None, aspect_ratios=[[0.5, 1.0, 2.0]]):
        super().__init__()
        self.strides = strides
        if sizes is None:
            sizes = [st * 4 for st in strides]  # базовые размеры
        self.sizes = sizes
        self.aspect_ratios = aspect_ratios
        self.num_anchors = len(aspect_ratios[0])

    def forward(self, features, image_size=640):
        anchors = []
        for level, (f, s, sz) in enumerate(zip(features, self.strides, self.sizes)):
            h, w = f.shape[2], f.shape[3]
            # сетка центров якорей
            ys = torch.arange(0, h, device=f.device) * s + s/2
            xs = torch.arange(0, w, device=f.device) * s + s/2
            ys, xs = torch.meshgrid(ys, xs, indexing='ij')
            cy = ys.reshape(-1)
            cx = xs.reshape(-1)
            # генерируем якоря с разными соотношениями сторон
            for ar in self.aspect_ratios[0]:
                w_anchor = sz * math.sqrt(ar)
                h_anchor = sz / math.sqrt(ar)
                x1 = cx - w_anchor/2
                y1 = cy - h_anchor/2
                x2 = cx + w_anchor/2
                y2 = cy + h_anchor/2
                anchors.append(torch.stack([x1, y1, x2, y2], dim=1))
        return torch.cat(anchors, dim=0)  # (total_anchors, 4)

### 5. Компоненты модели (из предыдущих ячеек с унификацией)

In [23]:
class Backbone(nn.Module):
    def __init__(self, backbone_name='resnet50', unfreeze_last=0, pretrained=True):
        super().__init__()
        weights = 'IMAGENET1K_V1' if pretrained else None
        resnet = torchvision.models.resnet50(weights=weights)
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4
        self.out_channels = [512, 1024, 2048]  # C3,C4,C5
        self._freeze_unfreeze(unfreeze_last)

    def _freeze_unfreeze(self, k):
        for param in self.parameters():
            param.requires_grad = False
        all_blocks = [self.layer1, self.layer2, self.layer3, self.layer4]
        for block in all_blocks[-k:]:
            for param in block.parameters():
                param.requires_grad = True

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        c2 = self.layer1(x)   # /4
        c3 = self.layer2(c2)  # /8
        c4 = self.layer3(c3)  # /16
        c5 = self.layer4(c4)  # /32
        return [c3, c4, c5]

class Neck(nn.Module):
    def __init__(self, in_channels_list, out_channels=256):
        """
        in_channels_list: список каналов от backbone в порядке [C3, C4, C5]
        """
        super().__init__()
        # Для top-down пути нужно начинать с C5 → C3, поэтому переворачиваем список
        rev_channels = in_channels_list[::-1]  # теперь [C5, C4, C3]
        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(c, out_channels, kernel_size=1) for c in rev_channels
        ])
        self.fpn_convs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
            for _ in rev_channels
        ])

    def forward(self, features):
        # features от backbone: [C3, C4, C5]
        features = features[::-1]  # [C5, C4, C3]
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]
        # Top-down с повышением разрешения
        x = laterals[0]                       # C5
        outs = [self.fpn_convs[0](x)]
        for i in range(1, len(laterals)):
            x = F.interpolate(x, scale_factor=2, mode='nearest') + laterals[i]
            outs.append(self.fpn_convs[i](x))
        # outs сейчас [P5, P4, P3] (низкое → высокое разрешение)
        # Возвращаем в порядке [P3, P4, P5]
        return outs[::-1]

class DecoupledHead(nn.Module):
    def __init__(self, in_channels, num_classes, num_anchors=1):
        super().__init__()
        self.num_classes = num_classes
        self.num_anchors = num_anchors
        self.cls_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
        )
        self.reg_conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, 3, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
        )
        self.cls_pred = nn.Conv2d(in_channels, num_anchors * num_classes, 1)
        self.reg_pred = nn.Conv2d(in_channels, num_anchors * 4, 1)
        self.obj_pred = nn.Conv2d(in_channels, num_anchors * 1, 1)

    def forward(self, x):
        cls_feat = self.cls_conv(x)
        reg_feat = self.reg_conv(x)
        cls = self.cls_pred(cls_feat)
        bbox = self.reg_pred(reg_feat)
        obj = self.obj_pred(reg_feat)
        return cls, bbox, obj

class Head(nn.Module):
    def __init__(self, in_channels, num_classes, num_anchors=1, num_levels=3):
        super().__init__()
        self.heads = nn.ModuleList([DecoupledHead(in_channels, num_classes, num_anchors) for _ in range(num_levels)])

    def forward(self, features):
        all_cls, all_bbox, all_obj = [], [], []
        for feat, head in zip(features, self.heads):
            c, b, o = head(feat)
            all_cls.append(c)
            all_bbox.append(b)
            all_obj.append(o)
        return all_cls, all_bbox, all_obj

### 6. Полный детектор с трейнингом

In [31]:
class Detector(nn.Module):
    def __init__(self, num_classes, backbone_unfreeze=2):
        super().__init__()
        self.num_classes = num_classes
        self.backbone = Backbone(unfreeze_last=backbone_unfreeze)
        self.neck = Neck(in_channels_list=self.backbone.out_channels, out_channels=256)
        self.anchor_gen = AnchorGenerator(
            strides=[8, 16, 32], sizes=[32, 64, 128],
            aspect_ratios=[[0.5, 1.0, 2.0]]
        )
        self.num_anchors_per_level = len(self.anchor_gen.aspect_ratios[0])
        self.head = Head(
            in_channels=256, num_classes=num_classes,
            num_anchors=self.num_anchors_per_level, num_levels=3
        )

    def forward(self, x):
        feats = self.backbone(x)
        feats = self.neck(feats)
        cls, bbox, obj = self.head(feats)
        return cls, bbox, obj, feats

    def get_anchors(self, img_size, device):
        """Возвращает якоря для всего изображения (total_anchors, 4)"""
        dummy = [
            torch.zeros(1, self.neck.fpn_convs[0].in_channels,
                        img_size // 8, img_size // 8, device=device),
            torch.zeros(1, self.neck.fpn_convs[0].in_channels,
                        img_size // 16, img_size // 16, device=device),
            torch.zeros(1, self.neck.fpn_convs[0].in_channels,
                        img_size // 32, img_size // 32, device=device)
        ]
        return self.anchor_gen(dummy, image_size=img_size)   # (total, 4)

    def decode_boxes(self, bbox_pred, anchors, img_size=640):
        """
        bbox_pred: (B, total, 4) - смещения
        anchors:   (total, 4)   - якоря xyxy
        возвращает: (B, total, 4) декодированные xyxy
        """
        a_w = anchors[:, 2] - anchors[:, 0]          # (total,)
        a_h = anchors[:, 3] - anchors[:, 1]          # (total,)
        a_cx = (anchors[:, 0] + anchors[:, 2]) / 2
        a_cy = (anchors[:, 1] + anchors[:, 3]) / 2

        dx = bbox_pred[:, :, 0] * a_w                # (B, total)
        dy = bbox_pred[:, :, 1] * a_h
        pred_cx = a_cx + dx
        pred_cy = a_cy + dy
        pred_w = a_w * torch.exp(bbox_pred[:, :, 2].clamp(max=4.0))
        pred_h = a_h * torch.exp(bbox_pred[:, :, 3].clamp(max=4.0))

        x1 = pred_cx - pred_w / 2
        y1 = pred_cy - pred_h / 2
        x2 = pred_cx + pred_w / 2
        y2 = pred_cy + pred_h / 2

        x1 = torch.clamp(x1, 0, img_size)
        y1 = torch.clamp(y1, 0, img_size)
        x2 = torch.clamp(x2, 0, img_size)
        y2 = torch.clamp(y2, 0, img_size)
        x2 = torch.max(x2, x1 + 1e-6)
        y2 = torch.max(y2, y1 + 1e-6)
        return torch.stack([x1, y1, x2, y2], dim=-1)

    def compute_loss(self, cls_preds, bbox_preds, obj_preds, feats, targets):
        B = cls_preds[0].size(0)
        device = cls_preds[0].device
        # Получаем все якоря (total, 4)
        anchors = self.get_anchors(img_size=cls_preds[0].shape[2] * 8, device=device)
        num_levels = len(cls_preds)

        # Собираем все предсказания в плоские тензоры (B, total, ...)
        all_cls = []
        all_bbox = []
        all_obj = []
        for lvl in range(num_levels):
            H, W = cls_preds[lvl].shape[2], cls_preds[lvl].shape[3]
            C = self.num_classes
            num_anchors = self.num_anchors_per_level
            cls = cls_preds[lvl].view(B, num_anchors, C, H, W).permute(0, 1, 3, 4, 2).reshape(B, -1, C)
            bbox = bbox_preds[lvl].view(B, num_anchors, 4, H, W).permute(0, 1, 3, 4, 2).reshape(B, -1, 4)
            obj = obj_preds[lvl].view(B, num_anchors, 1, H, W).permute(0, 1, 3, 4, 2).reshape(B, -1, 1)
            all_cls.append(cls)
            all_bbox.append(bbox)
            all_obj.append(obj.squeeze(-1))

        all_cls = torch.cat(all_cls, dim=1)      # (B, total, C)
        all_bbox = torch.cat(all_bbox, dim=1)    # (B, total, 4)
        all_obj = torch.cat(all_obj, dim=1)      # (B, total)

        # Декодированные боксы (xyxy) для подсчёта IoU
        all_bbox_xyxy = self.decode_boxes(all_bbox, anchors)   # ← ИСПРАВЛЕНО

        # Подготавливаем данные для TAL_assigner
        cls_for_tal = []
        bbox_for_tal = []
        for lvl in range(num_levels):
            H, W = cls_preds[lvl].shape[2], cls_preds[lvl].shape[3]
            cls_for_tal.append(cls_preds[lvl].view(B, self.num_anchors_per_level, self.num_classes, H, W))
            bbox_for_tal.append(bbox_preds[lvl].view(B, self.num_anchors_per_level, 4, H, W))

        anchors_per_level = [self.get_anchors_at_level(lvl, 640, device) for lvl in range(num_levels)]
        gt_boxes_list = [t['boxes'].to(device) for t in targets]
        gt_labels_list = [t['labels'].to(device) for t in targets]

        target_labels, target_boxes, pos_mask = TAL_assigner(
            cls_for_tal, bbox_for_tal, anchors_per_level,
            gt_boxes_list, gt_labels_list
        )

        # --- Лосс регрессии (DIoU) только для положительных якорей ---
        pos_idx = pos_mask
        if pos_idx.sum() > 0:
            pred_boxes_pos = all_bbox_xyxy[pos_idx]   # (N_pos, 4)
            gt_boxes_pos = target_boxes[pos_idx]
            loss_box = diou_loss(pred_boxes_pos, gt_boxes_pos)
        else:
            loss_box = all_bbox_xyxy.sum() * 0.0

        # --- Лосс классификации (Focal Loss) только для положительных ---
        cls_target = target_labels.clone()   # (B, total), -1 = фон/игнор
        if pos_idx.sum() > 0:
            pred_cls_pos = all_cls[pos_idx]            # (N_pos, C)
            gt_cls_pos = cls_target[pos_idx].long()   # (N_pos,)
            alpha = 0.25
            gamma = 2.0
            ce_loss = F.cross_entropy(pred_cls_pos, gt_cls_pos, reduction='none')
            pt = torch.exp(-ce_loss)
            loss_cls = (alpha * (1 - pt) ** gamma * ce_loss).mean()
        else:
            loss_cls = all_cls.sum() * 0.0

        # --- Лосс объектности (BCE) ---
        obj_label = pos_mask.float()
        loss_obj = F.binary_cross_entropy_with_logits(all_obj, obj_label, reduction='mean')

        return loss_box, loss_cls, loss_obj

    def get_anchors_at_level(self, level, img_size, device):
        """Возвращает якоря одного уровня (N_level, 4)"""
        dummy = [
            torch.zeros(1, 256, img_size // 8, img_size // 8, device=device),
            torch.zeros(1, 256, img_size // 16, img_size // 16, device=device),
            torch.zeros(1, 256, img_size // 32, img_size // 32, device=device)
        ]
        all_anchors = self.anchor_gen(dummy, image_size=img_size)  # (total, 4)
        H, W = dummy[level].shape[2], dummy[level].shape[3]
        num_anch = self.num_anchors_per_level
        start = level * H * W * num_anch
        end = start + H * W * num_anch
        return all_anchors[start:end]


# Переопределим TAL_assigner для использования с предварительно собранным тензором вероятностей
def TAL_assigner(cls_list, bbox_list, anchors_per_level, gt_boxes_list, gt_labels_list,
                 alpha=6.0, beta=1.0, top_k=5, bg_class=-1):
    """
    bg_class = -1 означает, что фон не считается отдельным классом.
    Для отрицательных якорей мы будем ставить ignore (-1), чтобы они не участвовали в лоссе классификации.
    """
    B = cls_list[0].size(0)
    device = cls_list[0].device
    num_anch_types = cls_list[0].size(1)
    C = cls_list[0].size(2)  # реальное число классов!

    all_cls_prob = []
    all_bbox_deltas = []
    all_anch = []

    for lvl, anchors in enumerate(anchors_per_level):
        cls = cls_list[lvl]      # (B, num_anch, C, H, W)
        bbox = bbox_list[lvl]    # (B, num_anch, 4, H, W)
        B_, N_anch, C_, H, W = cls.shape

        cls_prob = torch.sigmoid(
            cls.permute(0, 1, 3, 4, 2).reshape(B_, N_anch * H * W, C_)
        )
        bbox_delta = bbox.permute(0, 1, 3, 4, 2).reshape(B_, N_anch * H * W, 4)
        anchors_batch = anchors.unsqueeze(0).expand(B_, -1, -1)  # (B, N_lvl, 4)

        all_cls_prob.append(cls_prob)
        all_bbox_deltas.append(bbox_delta)
        all_anch.append(anchors_batch)

    all_cls_prob = torch.cat(all_cls_prob, dim=1)      # (B, total, C)
    all_bbox_deltas = torch.cat(all_bbox_deltas, dim=1) # (B, total, 4)
    all_anchors = torch.cat(all_anch, dim=1)             # (B, total, 4)

    # Декодируем боксы (xyxy) в пределах 640
    def decode_boxes_static(bbox_deltas, anchors_tensor, img_size=640):
        a_w = anchors_tensor[..., 2] - anchors_tensor[..., 0]
        a_h = anchors_tensor[..., 3] - anchors_tensor[..., 1]
        a_cx = (anchors_tensor[..., 0] + anchors_tensor[..., 2]) / 2
        a_cy = (anchors_tensor[..., 1] + anchors_tensor[..., 3]) / 2

        dx = bbox_deltas[..., 0] * a_w
        dy = bbox_deltas[..., 1] * a_h
        pred_cx = a_cx + dx
        pred_cy = a_cy + dy
        pred_w = a_w * torch.exp(bbox_deltas[..., 2].clamp(max=4.0))
        pred_h = a_h * torch.exp(bbox_deltas[..., 3].clamp(max=4.0))

        x1 = pred_cx - pred_w / 2
        y1 = pred_cy - pred_h / 2
        x2 = pred_cx + pred_w / 2
        y2 = pred_cy + pred_h / 2

        x1 = torch.clamp(x1, 0, img_size)
        y1 = torch.clamp(y1, 0, img_size)
        x2 = torch.clamp(x2, 0, img_size)
        y2 = torch.clamp(y2, 0, img_size)
        x2 = torch.max(x2, x1 + 1e-6)
        y2 = torch.max(y2, y1 + 1e-6)
        return torch.stack([x1, y1, x2, y2], dim=-1)

    pred_boxes = decode_boxes_static(all_bbox_deltas, all_anchors)

    total_anchors = pred_boxes.shape[1]
    # Инициализируем таргеты: для меток классов -1 = ignore, для боксов нули
    target_labels = torch.full((B, total_anchors), bg_class, device=device, dtype=torch.long)
    target_boxes = torch.zeros(B, total_anchors, 4, device=device)
    pos_mask = torch.zeros(B, total_anchors, device=device, dtype=torch.bool)

    for i in range(B):
        gt_box = gt_boxes_list[i]
        gt_label = gt_labels_list[i]
        if len(gt_box) == 0:
            continue
        gt_box = torch.clamp(gt_box, 0, 640)
        gt_box[:, 2] = torch.max(gt_box[:, 2], gt_box[:, 0] + 1e-6)
        gt_box[:, 3] = torch.max(gt_box[:, 3], gt_box[:, 1] + 1e-6)

        M = gt_box.size(0)
        cls_prob = all_cls_prob[i]                         # (total, C)
        gt_cls_prob = cls_prob[:, gt_label.long()]         # (total, M)  ← вот здесь индексы не должны выходить за C
        ious = box_iou(pred_boxes[i], gt_box)              # (total, M)

        t = gt_cls_prob.pow(alpha) * ious.pow(beta)

        a_cx = (all_anchors[i, :, 0] + all_anchors[i, :, 2]) / 2
        a_cy = (all_anchors[i, :, 1] + all_anchors[i, :, 3]) / 2
        inside_mask = torch.zeros(total_anchors, M, dtype=torch.bool, device=device)
        for j in range(M):
            gt = gt_box[j]
            inside_mask[:, j] = (a_cx >= gt[0]) & (a_cx <= gt[2]) & (a_cy >= gt[1]) & (a_cy <= gt[3])
        t[~inside_mask] = -1e9

        assigned_gt_idx = torch.full((total_anchors,), -1, device=device, dtype=torch.long)
        for j in range(M):
            vals, idx = torch.topk(t[:, j], top_k)
            for k in idx:
                if assigned_gt_idx[k] == -1:
                    assigned_gt_idx[k] = j
                else:
                    prev = assigned_gt_idx[k]
                    if ious[k, j] > ious[k, prev]:
                        assigned_gt_idx[k] = j

        pos = assigned_gt_idx >= 0
        pos_mask[i] = pos
        assigned_gt = assigned_gt_idx[pos]
        target_labels[i, pos] = gt_label[assigned_gt]   # реальные метки классов
        target_boxes[i, pos] = gt_box[assigned_gt]

    return target_labels, target_boxes, pos_mask

### 7. Функция фильтрации предсказаний (NMS)

In [38]:
@torch.no_grad()
def filter_predictions(cls, bbox, obj, anchors,
                       num_classes, num_anchors_per_level,
                       score_threshold=0.1, nms_threshold=0.5, img_size=640):
    """
    cls, bbox, obj – списки тензоров с разных уровней FPN
    anchors – все якоря (total_anchors, 4)
    num_classes – количество классов (int)
    num_anchors_per_level – число якорей на каждой клетке (int)
    """
    B = cls[0].size(0)
    device = cls[0].device
    all_results = []

    cls_list = []
    bbox_list = []
    obj_list = []
    for lvl in range(len(cls)):
        C = num_classes
        N_anch = num_anchors_per_level
        H, W = cls[lvl].shape[2], cls[lvl].shape[3]

        # cls[lvl]: (B, N_anch*C, H, W) → (B, N_anch, C, H, W) → (B, N_anch*H*W, C)
        cls_ = cls[lvl].view(B, N_anch, C, H, W).permute(0,1,3,4,2).reshape(B, -1, C)
        # bbox[lvl]: (B, N_anch*4, H, W) → (B, N_anch, 4, H, W) → (B, N_anch*H*W, 4)
        bbox_ = bbox[lvl].view(B, N_anch, 4, H, W).permute(0,1,3,4,2).reshape(B, -1, 4)
        # obj[lvl]: (B, N_anch*1, H, W) → (B, N_anch, 1, H, W) → (B, N_anch*H*W, 1)
        obj_ = obj[lvl].view(B, N_anch, 1, H, W).permute(0,1,3,4,2).reshape(B, -1, 1)

        cls_list.append(cls_)
        bbox_list.append(bbox_)
        obj_list.append(obj_)

    cls = torch.cat(cls_list, dim=1)      # (B, total_anchors, C)
    bbox = torch.cat(bbox_list, dim=1)    # (B, total_anchors, 4)
    obj = torch.cat(obj_list, dim=1).squeeze(-1)  # (B, total_anchors)

    # Декодирование предсказанных смещений в xyxy
    anchors_exp = anchors.unsqueeze(0).expand(B, -1, -1)
    a_w = anchors_exp[:, :, 2] - anchors_exp[:, :, 0]
    a_h = anchors_exp[:, :, 3] - anchors_exp[:, :, 1]
    a_cx = (anchors_exp[:, :, 0] + anchors_exp[:, :, 2]) / 2
    a_cy = (anchors_exp[:, :, 1] + anchors_exp[:, :, 3]) / 2

    dx = bbox[:, :, 0] * a_w
    dy = bbox[:, :, 1] * a_h
    pred_cx = a_cx + dx
    pred_cy = a_cy + dy
    pred_w = a_w * torch.exp(bbox[:, :, 2])
    pred_h = a_h * torch.exp(bbox[:, :, 3])

    x1 = pred_cx - pred_w / 2
    y1 = pred_cy - pred_h / 2
    x2 = pred_cx + pred_w / 2
    y2 = pred_cy + pred_h / 2

    boxes_xyxy = torch.stack([x1, y1, x2, y2], dim=-1)   # (B, total_anchors, 4)

    # Итоговый confidence = objectness * max(class_prob)
    scores = torch.sigmoid(obj) * torch.sigmoid(cls).max(dim=2)[0]   # (B, total_anchors)
    labels = torch.sigmoid(cls).argmax(dim=2)                        # (B, total_anchors)

    for i in range(B):
        keep = scores[i] > score_threshold
        if keep.sum() == 0:
            all_results.append({
                'boxes': torch.zeros((0, 4), device=device),
                'scores': torch.zeros(0),
                'labels': torch.zeros(0, dtype=torch.long)
            })
            continue

        bx = boxes_xyxy[i][keep]
        sc = scores[i][keep]
        lb = labels[i][keep]

        nms_idx = nms(bx, sc, nms_threshold)
        all_results.append({
            'boxes': bx[nms_idx].cpu(),
            'scores': sc[nms_idx].cpu(),
            'labels': lb[nms_idx].cpu()
        })

    return all_results

In [28]:
# Вставляем сразу после создания train_dataset, test_dataset
all_labels = set()
for _, target in train_dataset:
    all_labels.update(target['labels'].tolist())
num_classes = max(all_labels) + 1
print(f"Найдено классов: {num_classes}, метки в датасете: {sorted(all_labels)}")

Найдено классов: 4, метки в датасете: [0, 1, 2, 3]


### 8. Train/Val цикл

In [39]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, targets in tqdm(loader, desc='Train'):
        images = images.to(device)
        cls_preds, bbox_preds, obj_preds, feats = model(images)
        loss_box, loss_cls, loss_obj = model.compute_loss(cls_preds, bbox_preds, obj_preds, feats, targets)
        loss = loss_box + loss_cls + loss_obj
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader, device, score_threshold=0.1, nms_threshold=0.5):
    model.eval()
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    anchors = model.get_anchors(640, device)

    for images, targets in tqdm(loader, desc='Val'):
        images = images.to(device)
        cls_preds, bbox_preds, obj_preds, _ = model(images)
        preds = filter_predictions(
            cls_preds, bbox_preds, obj_preds, anchors,
            num_classes=model.num_classes,
            num_anchors_per_level=model.num_anchors_per_level,
            score_threshold=score_threshold,
            nms_threshold=nms_threshold,
            img_size=640
        )

        targets_cpu = [{k: v.cpu() for k, v in t.items() if k != 'image_id'} for t in targets]
        metric.update(preds, targets_cpu)

    return metric.compute()['map'].item()

### 9. Запуск обучения

In [40]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = Detector(num_classes=num_classes, backbone_unfreeze=2).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,
                          collate_fn=collate_fn, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False,
                         collate_fn=collate_fn, num_workers=2)

num_epochs = 20
best_map = 0.0
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    mAP = validate(model, test_loader, device, score_threshold=0.1, nms_threshold=0.5)
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {train_loss:.8f} | mAP: {mAP:.8f}")
    if mAP > best_map:
        best_map = mAP
        torch.save(model.state_dict(), 'best_detector.pth')


print(f"Лучший mAP: {best_map:.8f}")

Val:   0%|          | 0/34 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.
  warnings.warn(*args, **kwargs)
Val: 100%|██████████| 34/34 [00:08<00:00,  4.16it/s]


Epoch 1/20 | Loss: 1.6154 | mAP: 0.0000


Val: 100%|██████████| 34/34 [00:06<00:00,  5.32it/s]


Epoch 2/20 | Loss: 1.2840 | mAP: 0.0000


Val: 100%|██████████| 34/34 [00:05<00:00,  5.75it/s]


...

Epoch 18/20 | Loss: 0.9992 | mAP: 0.0000


Val: 100%|██████████| 34/34 [00:06<00:00,  5.67it/s]


Epoch 19/20 | Loss: 0.9930 | mAP: 0.0000


Val: 100%|██████████| 34/34 [00:05<00:00,  5.70it/s]

Epoch 20/20 | Loss: 0.9900 | mAP: 0.0000
Лучший mAP: 0.0000


In [41]:
print(best_map)

2.7780555456047296e-07


Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [19]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 28.0 MB/s eta 0:00:00


In [34]:
from torchmetrics.detection import MeanAveragePrecision

@torch.no_grad()
def validate(dataloader, filter_predictions_func, box_format="xyxy", device="cpu", score_threshold=0.1, nms_threshold=0.5, **kwargs):
    """ Метод для валидации модели.
    Возвращает mAP (0.5 ... 0.95).
    """
    self.model.eval()
    # Считаем метрику mAP с помощью функции из torchmetrics
    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")
    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)
        outputs = self.model(images)
        predicts = filter_predictions_func(outputs, score_threshold, nms_threshold, **kwargs)
        metric.update(predicts, targets)
    return metric.compute()["map"].item()
